In [ ]:
from contextlib import nullcontext
from pathlib import Path

import torch
from plotly.subplots import make_subplots
from tqdm.autonotebook import tqdm

from src.data.gaussian_mixture.data import MultimodalGaussianDataConfig
from src.drifting.model import DriftingModelConfig
from src.drifting.study import DriftingStudyConfig, DriftingStudyState
from src.drifting.transport import GaussianKernelConfig, ReverseKLDriftingLossConfig
from src.drifting.visualization import (
    RegularGridConfig,
    make_drifting_figure,
    make_regular_grid,
)
from src.model.mlp import StackedResidualMLPConfig
from src.monitoring.utils import save_go_detached
from src.priors.unimodal import UnimodalGaussianPriorConfig

torch.manual_seed(0)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
artifact_root = Path("artifacts/drifting")
batch_size = 1024
visualize_batch_size = 1024
num_train_steps = 500
snapshot_every = 50
save_threads = []


def get_device_context():
    if device.type == "cuda":
        return torch.cuda.device(device)
    return nullcontext()


def get_autocast_context():
    if device.type == "cuda":
        return torch.autocast(device_type=device.type, dtype=torch.bfloat16)
    return nullcontext()
    


In [ ]:
grid_config = RegularGridConfig(
    x_range=(-0.75, 3.25),
    y_range=(-2.0, 2.0),
    num_points_per_axis=81,
    quiver_stride=4,
    model_quiver_stride=10,
)
study_config = DriftingStudyConfig(
    data=MultimodalGaussianDataConfig.initialize(
        num_modes=2,
        mode_radius=1.25,
        mode_std=0.12,
        ambient_dimension=2,
        offset=1.25,
    ),
    prior=UnimodalGaussianPriorConfig.initialize(latent_shape=[2]),
    model=DriftingModelConfig.initialize(
        generator=StackedResidualMLPConfig.initialize(
            layer_dims=[2, 32, 32, 2],
        ),
        lr=1e-3,
        grad_clip_norm=1.0,
        linear_weight_scale=0.2,
        zero_linear_bias=True,
        output_bias=[0.5, 0.1],
    ),
    drifting=ReverseKLDriftingLossConfig(
        kernel=GaussianKernelConfig(width=0.35),
        transport_step_multiplier=1.0,
        transport_step_cap=0.12,
    ),
)
artifact_root.mkdir(parents=True, exist_ok=True)
display(study_config.visualize())
state = DriftingStudyState.initialize(
    config=study_config,
    device=device,
)
    


### Monitoring
    


In [ ]:
monitor_data = state.config.data.sample_unconditional(batch_size=visualize_batch_size).to(device)
monitor_latent = state.sample_prior(batch_size=visualize_batch_size)
mode_centers = state.config.data.mode_centers().to(device)
grid_points, grid_x, grid_y = make_regular_grid(
    config=grid_config,
    device=device,
    dtype=torch.float32,
)


def build_snapshot(state, *, step: int):
    with torch.no_grad():
        model_samples = state.decode(monitor_latent)
        grid_field_estimate = state.config.drifting.estimate_reverse_kl_field(
            query_points=grid_points,
            data_samples=monitor_data,
            model_samples=model_samples,
        )
        model_field_estimate = state.config.drifting.estimate_reverse_kl_field(
            query_points=model_samples,
            data_samples=monitor_data,
            model_samples=model_samples,
        )
        representative_model_samples = model_samples[:: grid_config.model_quiver_stride]
        representative_model_field = state.config.drifting.scale_clip_transport_field(
            model_field_estimate.reverse_kl,
        )[:: grid_config.model_quiver_stride]
        scaled_grid_field = state.config.drifting.scale_clip_transport_field(
            grid_field_estimate.reverse_kl,
        ).reshape(grid_x.shape[0], grid_x.shape[1], 2)
        model_density = grid_field_estimate.model_density.reshape(
            grid_x.shape[0],
            grid_x.shape[1],
        )
        nearest_mode_counts = torch.bincount(
            torch.cdist(model_samples, mode_centers).argmin(dim=1),
            minlength=2,
        )
    return {
        "step": step,
        "data_samples": monitor_data.detach().cpu(),
        "model_samples": model_samples.detach().cpu(),
        "representative_model_samples": representative_model_samples.detach().cpu(),
        "representative_model_field": representative_model_field.detach().cpu(),
        "grid_x": grid_x.detach().cpu(),
        "grid_y": grid_y.detach().cpu(),
        "model_density": model_density.detach().cpu(),
        "scaled_grid_field": scaled_grid_field.detach().cpu(),
        "nearest_mode_counts": nearest_mode_counts.detach().cpu(),
    }


def figure_from_snapshot(snapshot):
    return make_drifting_figure(
        data_samples=snapshot["data_samples"],
        model_samples=snapshot["model_samples"],
        representative_model_samples=snapshot["representative_model_samples"],
        representative_model_field=snapshot["representative_model_field"],
        grid_x=snapshot["grid_x"],
        grid_y=snapshot["grid_y"],
        model_density=snapshot["model_density"],
        grid_vector_field=snapshot["scaled_grid_field"],
        quiver_stride=grid_config.quiver_stride,
    )


def monitor(state, *, step: int):
    return figure_from_snapshot(build_snapshot(state, step=step))


def save_snapshot(state, *, step: int, name: str):
    snapshot = build_snapshot(state, step=step)
    fig = figure_from_snapshot(snapshot)
    torch.save(snapshot, artifact_root / f"{name}.pt")
    save_threads.append(save_go_detached(fig, folder=artifact_root, name=name))
    return fig


def make_progress_figure(snapshot_names):
    combo = make_subplots(
        rows=1,
        cols=len(snapshot_names),
        horizontal_spacing=0.03,
    )
    for column, name in enumerate(snapshot_names, start=1):
        snapshot = torch.load(artifact_root / f"{name}.pt", weights_only=False)
        fig = figure_from_snapshot(snapshot)
        for trace in fig.data:
            trace.showlegend = False
            combo.add_trace(trace, row=1, col=column)
        combo.update_xaxes(
            visible=False,
            range=list(grid_config.x_range),
            row=1,
            col=column,
            scaleanchor="y" if column == 1 else f"y{column}",
        )
        combo.update_yaxes(
            visible=False,
            range=list(grid_config.y_range),
            row=1,
            col=column,
        )
    combo.update_layout(
        template="plotly_white",
        width=700 * len(snapshot_names),
        height=700,
        margin=dict(l=10, r=10, t=10, b=10),
        showlegend=False,
    )
    return combo
    


### Reverse-KL drifting

This uses an MLP generator with deliberately small initial weights and a slight positive output bias. The two data modes sit near `(0, 0)` and `(2.5, 0)`. Because the initial pushforward is unimodal and much closer to the left basin, reverse-KL drifting is happy to commit to that single mode instead of covering both.
    


In [ ]:
initial_figure = save_snapshot(state, step=0, name="0000")
initial_figure

pbar = tqdm(range(num_train_steps))
for step in pbar:
    data_samples = state.config.data.sample_unconditional(batch_size=batch_size).to(device)

    with get_device_context(), get_autocast_context():
        loss = state.compute_drifting_loss(data_samples=data_samples)
        total_loss = loss.sum()

    total_loss.backward()
    state.step_and_zero_grad()

    with torch.no_grad():
        monitor_model_samples = state.decode(monitor_latent)
        nearest_mode_counts = torch.bincount(
            torch.cdist(monitor_model_samples, mode_centers).argmin(dim=1),
            minlength=2,
        )
    pbar.set_postfix(
        {
            "loss": f"{total_loss.item():.5f}",
            "left": int(nearest_mode_counts[0].item()),
            "right": int(nearest_mode_counts[1].item()),
        }
    )

    if (step + 1) % snapshot_every == 0:
        save_snapshot(
            state,
            step=step + 1,
            name=f"{step + 1:04d}",
        )

for thread in save_threads:
    thread.join()

torch.save(state, artifact_root / "state.pkl")
monitor(state, step=num_train_steps)
    


In [ ]:
state = torch.load(
    artifact_root / "state.pkl",
    map_location=device,
    weights_only=False,
)
    


## Progression and final pushforward
    


In [ ]:
available_snapshots = sorted(path.stem for path in artifact_root.glob("*.pt"))
selected_snapshots = [
    available_snapshots[0],
    available_snapshots[len(available_snapshots) // 2],
    available_snapshots[-1],
]

progress_figure = make_progress_figure(selected_snapshots)
progress_figure
    
